# SK_like Calibration Source Visualization

Two calibration events on the SK_like geometry:
1. A laser fibre at the top of the tank pointing downward.
2. An off-centre isotropic point source.

Source pattern follows `good_notebooks/laser_source_grad_analysis.ipynb`;
visualization follows `good_notebooks/geometry_and_events_3D_visualization.ipynb`.

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import torch

import jax
import jax.numpy as jnp

from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import laser_source, isotropic_source, load_detector_params
from lucid.utils import save_single_event, load_single_event
from lucid.visualization import create_detector_display

In [ ]:
GEOM    = '../config/SK_like_geom_config.json'
PHYSICS = '../config/SK_like_physics_config.json'

detector        = generate_detector(GEOM)
detector_params = load_detector_params(PHYSICS)

Path('output').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)

print(f'SK_like: H = {detector.H:.2f} m, r = {detector.r:.2f} m, '
      f'{len(detector.all_points)} sensors')

In [ ]:
Nphot = 10_000_000

simulator = setup_event_simulator(
    GEOM, Nphot,
    temperature=0.0, K=6,
    is_data=False, is_calibration=True,
    detector_type='Cylinder',
    max_candidates_per_ray=4,
    physics_config=PHYSICS,
    default_detector_params=True,
    hit_mode='aggregated',
)

# 2D unrolled-cylinder display (barrel + top/bottom caps on a flat canvas).
display_2d = create_detector_display(GEOM, sparse=True)

## 1. Laser fibre at the top of the tank

In [ ]:
# Fibre at the top of the tank, pointing down (default direction = [0,0,-1]).
# fiber_NA=0.22 is the same value used in laser_source_grad_analysis.
laser = laser_source(
    position=[0.0, 0.0, detector.H / 2 - 0.1],
    intensity=1e8,
    fiber_NA=0.22,
)

laser_event = jax.lax.stop_gradient(simulator(laser, jax.random.PRNGKey(0)))

save_single_event(
    laser_event, laser, detector_params,
    filename='output/SK_like_laser_event.h5',
    calibration_mode=True,
)

print(f'Active sensors: {int(jnp.sum(laser_event[0] > 0))}')
print(f'Total charge:   {float(jnp.sum(laser_event[0])):.1f}')

In [ ]:
_, _, indices, charges, times = load_single_event(
    'output/SK_like_laser_event.h5', None, calibration_mode=True,
)

# detector.visualize_event_data_plotly_discs(
#     indices, charges, times,
#     show_all_sensors=True,
#     log_scale=True,
#     show_colorbar=True,
#     dark_theme=False,
#     plot_time=False,
#     colorscale='viridis',
#     surface_color='black',
#     figname='figures/SK_like_laser_event.pdf',
# )

In [ ]:
# 2D unrolled view of the same laser event.
display_2d(
    indices, charges, times,
    log_scale=True,
    plot_time=False,
    file_name='figures/SK_like_laser_event_2D.pdf',
)

## 2. Off-centre isotropic point source

Place a point source half-way out toward the barrel and half-way up
toward the top cap. The isotropic generator emits uniformly over the
full sphere, so the illumination pattern is asymmetric across the
detector — bright on the near barrel/cap, dim on the far side.

In [ ]:
iso_position = [0.5 * detector.r, 0.0, 0.25 * detector.H]

iso = isotropic_source(
    position=iso_position,
    intensity=1e9,
)

iso_event = jax.lax.stop_gradient(simulator(iso, jax.random.PRNGKey(1)))

save_single_event(
    iso_event, iso, detector_params,
    filename='output/SK_like_isotropic_event.h5',
    calibration_mode=True,
)

print(f'Source position: {iso_position}')
print(f'Active sensors:  {int(jnp.sum(iso_event[0] > 0))}')
print(f'Total charge:    {float(jnp.sum(iso_event[0])):.1f}')

In [ ]:
# 2D unrolled view of the same isotropic event.
display_2d(
    indices, charges, times,
    log_scale=True,
    plot_time=False,
    file_name='figures/SK_like_isotropic_event_2D.pdf',
)

In [ ]:
_, _, indices, charges, times = load_single_event(
    'output/SK_like_isotropic_event.h5', None, calibration_mode=True,
)

# detector.visualize_event_data_plotly_discs(
#     indices, charges, times,
#     show_all_sensors=True,
#     log_scale=True,
#     show_colorbar=True,
#     dark_theme=False,
#     plot_time=False,
#     colorscale='viridis',
#     surface_color='black',
#     figname='figures/SK_like_isotropic_event.pdf',
# )

## Variations

- **Side-firing fibre** (barrel injection):
  `laser_source(position=[detector.r - 0.1, 0, 0], direction=[-1, 0, 0], intensity=1e8)`
- **Off-axis spot on the bottom cap**: shift `position[:2]` off the axis,
  keep `direction=[0, 0, -1]`.
- **Wavelength-aware run**: pass `wavelength=405.0` to `laser_source` /
  `isotropic_source` and `wavelength_mode=True` to `setup_event_simulator`.
- **Time-of-arrival plot**: re-run `visualize_event_data_plotly_discs`
  with `plot_time=True` to colour discs by hit time instead of charge.